In [ ]:
# Run this cell ONCE so matplotlib is NumPy 2.x compatible (uses pytorch-nightly path so it's loaded first).
import subprocess
import sys
target = "/storage/scratch1/5/sshrestha304/pytorch-nightly"
out = subprocess.run(
    [sys.executable, "-m", "pip", "install", f"--target={target}", "--upgrade", "matplotlib>=3.8"],
    capture_output=True,
    text=True,
)
print(out.stdout)
if out.stderr:
    print(out.stderr)
print("Exit code:", out.returncode)
if out.returncode == 0:
    print("Done. Restart kernel (Kernel -> Restart) then run Cell 1.")
else:
    print("Failed. In a terminal run: pip install --target=/storage/scratch1/5/sshrestha304/pytorch-nightly --upgrade 'matplotlib>=3.8'")

In [ ]:
# Cell 1: Imports + CUDA optimizations
# Use PyTorch nightly from scratch (supports sm_120 / Blackwell)
import os
os.environ['TORCH_NVSHMEM_DISABLE'] = '1'  # Required for PyTorch 2.12 nightly without nvshmem lib
import sys
sys.path.insert(0, '/storage/scratch1/5/sshrestha304/pytorch-nightly')

import warnings
warnings.filterwarnings('ignore', message='.*is not compatible with the current PyTorch installation.*', module='torch.cuda')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import os
import json
import time
from typing import List, Optional, Tuple
from PIL import Image

# CUDA optimizations
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')

print("Imports complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    ngpu = torch.cuda.device_count()
    print(f"GPUs: {ngpu} — {[torch.cuda.get_device_name(i) for i in range(ngpu)]}")
    print(f"GPU 0 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Cell 1.5: GPU Check (run this after Cell 1 to verify GPU is working)
import torch
print("="*60)
print("GPU STATUS CHECK")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    ngpu = torch.cuda.device_count()
    print(f"Number of GPUs: {ngpu}")
    for i in range(ngpu):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        cap = torch.cuda.get_device_capability(i)
        arch = f"sm_{cap[0]*10 + cap[1]}"
        print(f"    Compute capability: {cap[0]}.{cap[1]} ({arch})")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    print(f"\nArch list: {torch.cuda.get_arch_list()}")
    if 'sm_89' in torch.cuda.get_arch_list():
        print("✅ GPU (sm_89) is supported - will use GPU")
    elif 'sm_75' in torch.cuda.get_arch_list():
        print("✅ GPU (sm_75) is supported - will use GPU")
    else:
        print("⚠️  GPU compute capability not in arch list - will fall back to CPU")
else:
    print("⚠️  CUDA not available - will use CPU")
print("="*60)

In [ ]:
# Cell 2: Batch Top-K Sparsity Activation
class BatchTopK(nn.Module):
    """
    Batch Top-K sparsity: Keep only top k activations across entire batch.
    This enforces sparsity by zeroing out all but the top k values.
    """
    def __init__(self, k: int, k_fraction: bool = False):
        super().__init__()
        self.k = k
        self.k_fraction = k_fraction
    
    def forward(self, x):
        """
        Args:
            x: (batch, features) - flattened activations
        Returns:
            Sparse activations with only top-k values kept
        """
        batch_size, num_features = x.shape
        total_elements = batch_size * num_features
        
        # Determine actual k
        if self.k_fraction:
            actual_k = max(1, int(total_elements * self.k))
        else:
            actual_k = min(self.k, total_elements)
        
        # Flatten to find top-k across entire batch
        x_flat = x.view(-1)  # (batch * features,)
        
        # Get top-k values and indices
        topk_values, topk_indices = torch.topk(x_flat, actual_k, dim=0)
        
        # Create sparse tensor
        sparse_x = torch.zeros_like(x_flat)
        sparse_x[topk_indices] = topk_values
        
        # Reshape back
        return sparse_x.view(batch_size, num_features)

print("✅ BatchTopK class defined")

In [ ]:
# Cell 3: Dataset Class (CHANNEL-BY-CHANNEL)
class LanczosProteinDataset(Dataset):
    """
    Loads protein pair representations and resizes them to target_size using Lanczos interpolation.
    CHANNEL-BY-CHANNEL: Loads only ONE channel at a time to reduce memory.
    """
    def __init__(
        self,
        data_paths: List[str],
        channel_idx: int = 0,  # Single channel index (not a list)
        target_size: int = 342,
        layer: int = 47,
    ):
        self.data_paths = data_paths
        self.channel_idx = channel_idx
        self.target_size = target_size
        self.layer = layer
        self.protein_names = []
        self.original_shapes = []
        self.data = []
        
        for path in data_paths:
            arr = np.load(path)
            h, w, c = arr.shape
            self.original_shapes.append(arr.shape)
            
            # Extract only the specified channel: (H, W, 1)
            if channel_idx >= c:
                raise ValueError(f"Channel index {channel_idx} >= num channels {c} in {path}")
            arr_single_channel = arr[:, :, channel_idx:channel_idx+1]  # Keep dim: (H, W, 1)
            self.data.append(arr_single_channel)
            
            # Extract protein name from path (works for .../protein_id/protein_id_pair_block_{layer}.npy)
            base = os.path.basename(path)
            if base.endswith(f'_pair_block_{layer}.npy'):
                protein_name = base.replace(f'_pair_block_{layer}.npy', '')
            else:
                protein_name = os.path.basename(os.path.dirname(path))
            self.protein_names.append(protein_name)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        arr = self.data[idx].copy()  # (H, W, 1)
        h, w, c = arr.shape
        assert c == 1, f"Expected single channel, got {c}"
        
        # Normalize per sample to [-1, 1]
        max_val = np.abs(arr).max()
        if max_val > 1e-8:
            arr = arr / max_val
        
        # Interpolate the single channel using Lanczos
        img = Image.fromarray(arr[:, :, 0].astype(np.float32), mode="F")
        img_resized = img.resize(
            (self.target_size, self.target_size), Image.Resampling.LANCZOS
        )
        resized = np.array(img_resized)  # (H, W)
        
        # Convert to tensor: (H, W) -> (1, H, W) for single channel
        tensor = torch.from_numpy(resized).unsqueeze(0)  # (1, H, W)
        return tensor, idx, max_val

print("✅ LanczosProteinDataset class defined (CHANNEL-BY-CHANNEL)")

In [ ]:
# Cell 4: Sparse Autoencoder Model (WITH WEIGHT TYING)
class SparseAutoEncoder(nn.Module):
    """
    Sparse Autoencoder with large latent space and WEIGHT TYING.
    Decoder weights are transposes of encoder weights (cuts memory by 50%).
    Fully connected feed-forward only - no convolutions or pooling.
    """
    def __init__(
        self,
        input_dim: int,
        latent_dim: int,
        encoder_hidden_dims: Optional[List[int]] = None,
        decoder_hidden_dims: Optional[List[int]] = None,
        topk_k: int = 1000,
        topk_fraction: bool = False,
        use_batch_norm: bool = False,
        use_weight_tying: bool = True,  # Enable weight tying by default
    ):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.topk_k = topk_k
        self.topk_fraction = topk_fraction
        self.use_weight_tying = use_weight_tying
        
        # Default: no hidden layers (direct projection)
        if encoder_hidden_dims is None:
            encoder_hidden_dims = []
        if decoder_hidden_dims is None:
            decoder_hidden_dims = []
        
        # Store for use in forward pass
        self.encoder_hidden_dims = encoder_hidden_dims
        
        # Store encoder Linear layers separately for weight tying
        self.encoder_linears = nn.ModuleList()
        self.encoder_batch_norms = nn.ModuleList() if use_batch_norm else None
        self.encoder_activations = nn.ModuleList()
        
        prev_dim = input_dim
        
        # Build encoder layers
        for hidden_dim in encoder_hidden_dims:
            self.encoder_linears.append(nn.Linear(prev_dim, hidden_dim))
            if use_batch_norm:
                if self.encoder_batch_norms is None:
                    self.encoder_batch_norms = nn.ModuleList()
                self.encoder_batch_norms.append(nn.BatchNorm1d(hidden_dim))
            self.encoder_activations.append(BatchTopK(topk_k, topk_fraction))
            prev_dim = hidden_dim
        
        # Final encoder layer to latent
        self.encoder_linears.append(nn.Linear(prev_dim, latent_dim))
        self.encoder_activations.append(BatchTopK(topk_k, topk_fraction))
        
        # Decoder: Only need activations (if weight tying)
        if not use_weight_tying:
            # Build decoder normally (old way)
            decoder_layers = []
            prev_dim = latent_dim
            for hidden_dim in decoder_hidden_dims:
                decoder_layers.append(nn.Linear(prev_dim, hidden_dim))
                if use_batch_norm:
                    decoder_layers.append(nn.BatchNorm1d(hidden_dim))
                decoder_layers.append(BatchTopK(topk_k, topk_fraction))
                prev_dim = hidden_dim
            decoder_layers.append(nn.Linear(prev_dim, input_dim))
            decoder_layers.append(nn.Tanh())
            self.decoder = nn.Sequential(*decoder_layers)
        else:
            # Weight tying: decoder uses transposed encoder weights
            # Store decoder activations (reverse order of encoder)
            self.decoder_activations = nn.ModuleList()
            self.decoder_batch_norms = nn.ModuleList() if use_batch_norm else None
            
            # Decoder activations: one for each encoder hidden layer
            # (We'll apply activations between transposed weight applications)
            for _ in range(len(encoder_hidden_dims)):
                self.decoder_activations.append(BatchTopK(topk_k, topk_fraction))
                if use_batch_norm and self.decoder_batch_norms is not None:
                    # Batch norm dimensions will match reversed encoder hidden dims
                    # Note: dimensions determined dynamically in forward pass
                    pass
            
            self.final_tanh = nn.Tanh()
    
    def forward(self, x, return_latent=False):
        """
        Args:
            x: (batch, input_dim) - flattened input
            return_latent: Whether to return latent representation
        Returns:
            reconstructed: (batch, input_dim)
            latent: (batch, latent_dim) if return_latent=True
        """
        # ENCODER: Forward pass
        h = x
        for i, linear in enumerate(self.encoder_linears):
            h = linear(h)
            if self.encoder_batch_norms is not None and i < len(self.encoder_batch_norms):
                h = self.encoder_batch_norms[i](h)
            h = self.encoder_activations[i](h)
        
        latent = h
        
        # DECODER: Forward pass with weight tying
        if self.use_weight_tying:
            h = latent
            # Decoder uses transposed encoder weights in reverse order
            # Example with encoder_hidden=[1000, 3000]:
            # Encoder: W0 (2.1M→1000), W1 (1000→3000), W2 (3000→6000)
            # Decoder: W2^T (6000→3000), W1^T (3000→1000), W0^T (1000→2.1M)
            
            num_encoder_layers = len(self.encoder_linears)
            activation_idx = 0
            
            # Process layers in reverse order (from last encoder layer to first)
            # Skip the first layer (index 0) for now - we'll handle it separately
            for i in range(num_encoder_layers - 1, 0, -1):
                encoder_linear = self.encoder_linears[i]
                # Apply transposed weight: h @ W^T
                h = F.linear(h, encoder_linear.weight.t(), None)
                
                # Apply activation after each transposed layer (except before final tanh)
                if activation_idx < len(self.decoder_activations):
                    if self.decoder_batch_norms is not None and activation_idx < len(self.decoder_batch_norms):
                        h = self.decoder_batch_norms[activation_idx](h)
                    h = self.decoder_activations[activation_idx](h)
                    activation_idx += 1
            
            # Final layer: transpose of first encoder layer, then tanh
            h = F.linear(h, self.encoder_linears[0].weight.t(), None)
            h = self.final_tanh(h)
            reconstructed = h
        else:
            # Old way: use separate decoder
            reconstructed = self.decoder(latent)
        
        if return_latent:
            return reconstructed, latent
        return reconstructed

print("✅ SparseAutoEncoder class defined (WITH WEIGHT TYING)")

In [ ]:
# Cell 5: Configuration
# ========== RUN SELECTOR: change RUN_INDEX (0..5) for each of your 6 runs ==========
# Layer 47 only. 0-3: 16 ch each; 4 = 64-95 (4+5 combined); 5 = 96-127 (6+7 combined).
#   RUN_INDEX 0: channels 0-15   | RUN_INDEX 3: channels 48-63
#   RUN_INDEX 1: channels 16-31  | RUN_INDEX 4: channels 64-95  (combined)
#   RUN_INDEX 2: channels 32-47  | RUN_INDEX 5: channels 96-127 (combined)
RUN_INDEX = 0  # Run 1 -> 0, Run 2 -> 1, ... Run 6 -> 5

CHANNEL_RANGES = [  # 6 ranges: 0-3 as-is (16 each), 4 = 4+5, 5 = 6+7
    (0, 16), (16, 32), (32, 48), (48, 64),
    (64, 96),   # combined 4+5
    (96, 128),  # combined 6+7
]
assert 0 <= RUN_INDEX <= 5, "RUN_INDEX must be 0..5"

# Only train/test on these proteins; 50-50 split; layer 47 only.
PROTEIN_LIST = [
    '6tf4_A', '6wmk_A', '6wqc_A', '6zpp_A', '7ab3_E', '7ar0_B', '7atr_A', '7au7_A',
    '7awk_A', '7b0d_A', '7b1k_B', '7b1w_F', '7b26_C', '7b28_F', '7b29_A', '7b2a_A',
    '7b2o_A', '7b3a_A', '7b4q_A', '7b7t_A', '7bbz_A', '7bcz_A', '7bew_A', '7bhy_A',
    '7cjs_B', '7dcm_A', '7dfe_A', '7djy_A', '7dk9_A', '7dkk_A', '7dko_A', '7dmf_A',
    '7dms_A', '7dnu_A', '7don_B', '7dq9_A', '7dru_C', '7dtp_A', '7dup_A', '7dut_A',
    '7dvn_A', '7e0m_A', '7e2v_A', '7e37_A', '7e3z_A', '7ebt_B', '7ee3_C', '7ef6_A',
    '7eqx_A', '7esx_A', '7et8_A', '7ewu_B', '7ezg_A', '7f0h_A', '7f6e_B', '7f8a_A',
    '7fbh_A', '7fbp_B', '7fe3_A', '7ffa_A', '7fh3_A', '7fip_A', '7jiz_A', '7jt9_A',
    '7kdx_B', '7kik_A', '7kiu_A', '7kos_A', '7kqv_D', '7ksp_A', '7kua_A', '7kzh_A',
    '7l6j_A', '7l6y_A', '7l8n_A', '7lbu_A', '7lc5_A', '7lgr_A', '7ljh_A', '7lnu_B',
    '7lqm_D', '7lqn_A', '7ls0_B', '7lvz_D', '7m4n_A', '7mcc_A', '7mfi_A', '7mfw_B',
    '7mpz_A', '7mrk_D', '7mro_A', '7mu9_A', '7mwr_A', '7ndr_A', '7nl4_A', '7ntn_A',
    '7ny6_A', '7o49_F', '7obm_A', '7ofn_A', '7on9_A', '7ool_A', '7ouq_A', '7p3b_B',
    '7p3t_B', '7p82_C', '7pab_A', '7pbk_A', '7plb_B', '7poh_B', '7ppp_A', '7pq7_A',
    '7pqf_A', '7prd_A', '7puo_A', '7q03_A', '7q1b_A', '7q47_A', '7r84_A', '7rbw_A',
    '7rdt_A', '7re4_A', '7re6_A', '7s94_C', '7sf6_A', '7sir_A', '7spo_C', '7spp_C',
    '7stt_A', '7stz_C', '7swh_A', '7swk_B', '7sy9_A', '7t24_A', '7t5f_B', '7t71_A',
    '7t9w_B', '7ta9_A', '7tav_B', '7tbs_A', '7tcb_B', '7thw_A', '7tj1_D', '7tkv_A',
    '7v1v_A', '7v5y_B', '7vdy_A', '7vmu_A', '7vnb_A', '7vpf_A', '7vw0_B', '7w83_A',
    '7wbr_A', '7wgk_A',
]

# Blackwell (sm_120) / unsupported GPU: fall back to CPU and point to PyTorch upgrade
def _get_safe_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        arch_list = torch.cuda.get_arch_list()
        cap = torch.cuda.get_device_capability(0)
        arch = f"sm_{cap[0] * 10 + cap[1]}"
        if arch_list is not None and arch not in arch_list:
            print(f"\\n⚠️  GPU {torch.cuda.get_device_name(0)} (compute {arch}) is not supported by this PyTorch build.")
            print(f"   Supported: {arch_list}. Falling back to CPU.")
            print("   For GPU: upgrade to PyTorch 2.7+ or: pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128\\n")
            return 'cpu'
    except Exception as e:
        print(f"\\n⚠️  GPU check failed ({e}). Using CPU.\\n")
        return 'cpu'
    return 'cuda'
_device = _get_safe_device()

CONFIG = {
    'protein_dir': '/storage/scratch1/5/sshrestha304/Autoencoder/CompleteProteins',
    'protein_list': PROTEIN_LIST,  # Only train/test on these proteins
    'layer': 47,
    'layers': [47],  # Only layer 47
    'train_ratio': 0.5,   # 50-50 train/test
    'val_ratio': 0,      # No val set; use test for val metric
    'test_ratio': 0.5,
    'split_seed': 42,
    'target_size': 128,  # Reduced to 128 to make model memory-efficient
    'num_channels': 128,  # Total number of channels to process (channel-by-channel)
    'epochs': 200,
    # Cluster (1 node, 8x RTX 6000, 32 cores): batch_size and num_workers tuned for throughput
    'batch_size': 128,  # 8× A100 80GB; use 64 or 32 if OOM
    'num_workers': 8,   # DataLoader workers (32 cores → 8 is safe)
    'use_data_parallel': True,
    'channel_range': CHANNEL_RANGES[RUN_INDEX],  # Set by RUN_INDEX above (6-way split)
    'lr': 0.001,
    'weight_decay': 1e-5,
    'topk_k': 1000,  # Number of top activations to keep
    'topk_fraction': False,  # If True, topk_k is fraction of total elements
    # CHANNEL-BY-CHANNEL: Input is now 1 channel × 128 × 128 = 16,384 per sample
    # Much smaller than before! This allows for larger hidden layers.
    'encoder_hidden': [500, 1500],  # Gradually compress: 16k -> 500 -> 1500 -> latent
    'decoder_hidden': [1500, 500],  # Gradually expand: latent -> 1500 -> 500 -> 16k
    'latent_multiplier': 2.0,  # Latent will be 2x the last encoder hidden dim (3k)
    'use_batch_norm': False,
    'use_weight_tying': True,  # Enable weight tying (cuts memory by 50%)
    'use_mixed_precision': True,  # Use float16 (cuts memory by ~50%)
    'dtype': 'float16',  # Changed to float16 (bfloat16 not supported on all GPUs)
    'device': _device,
    'output_dir': 'sae_results',
}

print("="*80)
print("CONFIGURATION - SPARSE AUTOENCODER (CHANNEL-BY-CHANNEL)")
print("="*80)
ch_start, ch_end = CONFIG['channel_range']
print(f"  >>> This run: RUN_INDEX={RUN_INDEX} -> channels {ch_start}-{ch_end-1} (all 48 layers)")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print("="*80)

In [ ]:
# Cell 6: Load Data (CHANNEL-BY-CHANNEL SETUP) — layer 47 only, protein_list only, 50-50 train/test
protein_dir = CONFIG['protein_dir']
if not os.path.isdir(protein_dir):
    raise FileNotFoundError(f"protein_dir not found: {protein_dir}")

def get_data_for_layer(layer):
    """Use only CONFIG['protein_list']; 50-50 train/test; no val (use test for val metric). Returns (protein_files, train_paths, val_paths, test_paths)."""
    protein_files = []
    for pdir in CONFIG.get('protein_list', []):
        pfile = os.path.join(protein_dir, pdir, f"{pdir}_pair_block_{layer}.npy")
        if os.path.exists(pfile):
            protein_files.append(pfile)
    protein_files.sort()
    n = len(protein_files)
    if n == 0:
        return [], [], [], []
    n_train = int(n * CONFIG['train_ratio'])  # 50%
    n_val = int(n * CONFIG['val_ratio'])      # 0
    n_test = n - n_train - n_val              # 50%
    gen = torch.Generator().manual_seed(CONFIG['split_seed'])
    perm = torch.randperm(n, generator=gen)
    train_paths = [protein_files[i] for i in perm[:n_train]]
    val_paths = [protein_files[i] for i in perm[n_train:n_train + n_val]]  # empty when val_ratio=0
    test_paths = [protein_files[i] for i in perm[n_train + n_val:]]
    return protein_files, train_paths, val_paths, test_paths

# Initialize from layer 47 (only layer we train)
protein_files, train_paths, val_paths, test_paths = get_data_for_layer(47)
if len(val_paths) == 0:
    val_paths = test_paths  # Use test set as 'val' for metric and best-model selection
print(f"✅ Layer 47: {len(protein_files)} protein files (from protein_list), train={len(train_paths)} test={len(test_paths)}")
if len(protein_files) == 0:
    raise ValueError("No protein files found for layer 47. Check protein_dir and protein_list.")

first_arr = np.load(protein_files[0])
_, _, num_channels_actual = first_arr.shape
num_channels = min(CONFIG['num_channels'], num_channels_actual)
input_dim_per_channel = CONFIG['target_size'] * CONFIG['target_size']
if CONFIG['encoder_hidden']:
    latent_dim = int(CONFIG['encoder_hidden'][-1] * CONFIG['latent_multiplier'])
else:
    latent_dim = int(input_dim_per_channel * CONFIG['latent_multiplier'])
print(f"✅ Channels: {num_channels}, input_dim_per_channel: {input_dim_per_channel:,}, latent_dim: {latent_dim:,}")

def create_data_loaders_for_channel(channel_idx, train_paths, val_paths, layer):
    """Create train/val loaders for a specific channel and layer (CUDA-optimized)."""
    nw = CONFIG.get('num_workers', 4)
    use_cuda = CONFIG['device'] == 'cuda'
    train_dataset = LanczosProteinDataset(train_paths, channel_idx=channel_idx, target_size=CONFIG['target_size'], layer=layer)
    val_dataset = LanczosProteinDataset(val_paths, channel_idx=channel_idx, target_size=CONFIG['target_size'], layer=layer)
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=nw, pin_memory=use_cuda, persistent_workers=(nw > 0))
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=nw, pin_memory=use_cuda, persistent_workers=(nw > 0))
    return train_loader, val_loader

def create_test_loader_for_channel(channel_idx, test_paths, layer):
    """Create test loader for a specific channel and layer."""
    nw = CONFIG.get('num_workers', 4)
    use_cuda = CONFIG['device'] == 'cuda'
    test_dataset = LanczosProteinDataset(test_paths, channel_idx=channel_idx, target_size=CONFIG['target_size'], layer=layer)
    return DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=nw, pin_memory=use_cuda, persistent_workers=(nw > 0))

print(f"✅ Data helpers ready for {num_channels} channels, layer 47 only")

In [ ]:
# Cell 7: Helper Function to Create Model for Channel (+ DataParallel when multi-GPU)
def create_model_for_channel(input_dim, latent_dim):
    """Create a new model for a specific channel; wrap in DataParallel if multiple GPUs."""
    model = SparseAutoEncoder(
        input_dim=input_dim,
        latent_dim=latent_dim,
        encoder_hidden_dims=CONFIG['encoder_hidden'],
        decoder_hidden_dims=CONFIG['decoder_hidden'],
        topk_k=CONFIG['topk_k'],
        topk_fraction=CONFIG['topk_fraction'],
        use_batch_norm=CONFIG['use_batch_norm'],
        use_weight_tying=CONFIG['use_weight_tying'],
    ).to(CONFIG['device'])
    if CONFIG['device'] == 'cuda' and torch.cuda.is_available() and torch.cuda.device_count() > 1 and CONFIG.get('use_data_parallel', True):
        model = nn.DataParallel(model)
    return model

def get_state_dict_to_save(model):
    """Return state_dict (strip DataParallel wrapper so checkpoint loads in single-GPU)."""
    return model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

device = torch.device(CONFIG['device'])
print(f"✅ Using device: {device}")
if CONFIG['device'] == 'cuda' and torch.cuda.is_available() and torch.cuda.device_count() > 1:
    print(f"✅ DataParallel: {torch.cuda.device_count()} GPUs")
print(f"✅ Model creation helper ready")

In [ ]:
# Cell 8: Model Architecture Info (per channel)
print("="*80)
print("MODEL ARCHITECTURE (PER CHANNEL)")
print("="*80)
print(f"  Input per channel: {input_dim_per_channel:,} ({CONFIG['target_size']} × {CONFIG['target_size']})")
if CONFIG['encoder_hidden']:
    print(f"  Encoder hidden: {' → '.join(map(str, CONFIG['encoder_hidden']))}")
print(f"  Latent: {latent_dim:,} ({CONFIG['latent_multiplier']}x last encoder hidden)")
if CONFIG['decoder_hidden']:
    print(f"  Decoder hidden: {' → '.join(map(str, CONFIG['decoder_hidden']))}")
print(f"  Output per channel: {input_dim_per_channel:,}")
print(f"  Batch Top-K: k={CONFIG['topk_k']}, fraction={CONFIG['topk_fraction']}")
print(f"  Weight Tying: {CONFIG['use_weight_tying']}")
print(f"\n  Total Channels to Process: {num_channels}")
print(f"  Models: {num_channels} separate models (one per channel)")
print("="*80)

In [ ]:
# Cell 9: Training Configuration Info
# Note: Optimizer, scheduler, and criterion are created per-channel inside the training loop
print("="*80)
print("TRAINING CONFIGURATION")
print("="*80)
print("✅ Loss: MSE")
print("✅ Optimizer: AdamW")
print(f"✅ Learning Rate: {CONFIG['lr']}")
print(f"✅ Weight Decay: {CONFIG['weight_decay']}")
print(f"✅ Scheduler: OneCycleLR")
print(f"✅ Batch Size: {CONFIG['batch_size']}")
print(f"✅ Epochs: {CONFIG['epochs']}")
print(f"✅ Mixed Precision: {CONFIG['use_mixed_precision']}")
if CONFIG['use_mixed_precision']:
    print(f"✅ Dtype: {CONFIG['dtype']}")
print("="*80)
print("Note: Training setup (optimizer, scheduler) happens per-channel in the training loop.")
print("="*80)

In [ ]:
# Cell 10: Training Functions (WITH MIXED PRECISION)
from torch.amp import autocast, GradScaler

def train_epoch(model, dataloader, criterion, optimizer, scheduler, device, topk_k, epoch, 
                use_mixed_precision=False, dtype=torch.float16, scaler=None):
    """Train for one epoch with optional mixed precision."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch_data, batch_indices, max_vals in dataloader:
        batch_data = batch_data.to(device, non_blocking=True)
        batch_size, channels, height, width = batch_data.shape
        
        # Flatten: (batch, C, H, W) -> (batch, C*H*W)
        batch_data_flat = batch_data.view(batch_size, -1)
        
        optimizer.zero_grad()
        
        # Forward pass with mixed precision
        if use_mixed_precision:
            with autocast('cuda', dtype=dtype):
                reconstructed_flat = model(batch_data_flat)
                loss = criterion(reconstructed_flat, batch_data_flat)
            
            # Backward pass with gradient scaling
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            # Standard precision
            reconstructed_flat = model(batch_data_flat)
            loss = criterion(reconstructed_flat, batch_data_flat)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        if scheduler is not None:
            scheduler.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches if num_batches > 0 else 0.0


def evaluate(model, dataloader, criterion, device, use_mixed_precision=False, dtype=torch.float16):
    """Evaluate model with optional mixed precision."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch_data, batch_indices, max_vals in dataloader:
            batch_data = batch_data.to(device, non_blocking=True)
            batch_size, channels, height, width = batch_data.shape
            
            # Flatten
            batch_data_flat = batch_data.view(batch_size, -1)
            
            # Forward pass with mixed precision
            if use_mixed_precision:
                with autocast('cuda', dtype=dtype):
                    reconstructed_flat = model(batch_data_flat)
                    loss = criterion(reconstructed_flat, batch_data_flat)
            else:
                reconstructed_flat = model(batch_data_flat)
                loss = criterion(reconstructed_flat, batch_data_flat)
            
            total_loss += loss.item()
            num_batches += 1
    
    return total_loss / num_batches if num_batches > 0 else 0.0

print("✅ Training functions defined (WITH MIXED PRECISION SUPPORT)")

In [ ]:
# Cell 11: Training Loop — ALL LAYERS 0..47, then CHANNEL-BY-CHANNEL
os.makedirs(CONFIG['output_dir'], exist_ok=True)

use_mixed_precision = CONFIG.get('use_mixed_precision', False)
if use_mixed_precision:
    dtype_str = CONFIG.get('dtype', 'float16')
    dtype = torch.bfloat16 if dtype_str == 'bfloat16' else torch.float16
    scaler = GradScaler('cuda') if dtype == torch.float16 else None
    print(f"✅ Mixed precision enabled: {dtype_str}")
else:
    dtype = torch.float32
    scaler = None
    print("✅ Using full precision (float32)")

# Per-layer results (for last layer we keep all_channel_results for backward compat with plot/save cells)
all_results_by_layer = {}

print("\n" + "="*80)
print("STARTING TRAINING — ALL LAYERS 0..47, CHANNEL-BY-CHANNEL")
print("="*80)
print(f"Layers: {CONFIG['layers'][:5]}...{CONFIG['layers'][-3:]}, {num_channels} channels each")
print("="*80)

overall_start_time = time.time()
_ch_start, _ch_end = CONFIG.get('channel_range', (0, num_channels))
_ch_end = min(_ch_end, num_channels)
total_channels_estimate = len(CONFIG['layers']) * (_ch_end - _ch_start)
channels_done = 0

for current_layer in CONFIG['layers']:
    protein_files_layer, train_paths, val_paths, test_paths = get_data_for_layer(current_layer)
    if len(val_paths) == 0:
        val_paths = list(test_paths)  # Use test as val for metric when val_ratio=0
    if len(protein_files_layer) == 0:
        print(f"⏭️  Layer {current_layer}: no files, skipping")
        continue

    output_dir_layer = os.path.join(CONFIG['output_dir'], f"layer_{current_layer}")
    os.makedirs(output_dir_layer, exist_ok=True)

    all_channel_results = {
        'train_losses': [], 'val_losses': [], 'best_val_losses': [],
        'final_train_losses': [], 'final_val_losses': [],
    }

    print(f"\n{'#'*80}")
    print(f"# LAYER {current_layer} — {len(protein_files_layer)} proteins, train={len(train_paths)} val={len(val_paths)} test={len(test_paths)}")
    print(f"{'#'*80}\n")

    ch_start, ch_end = CONFIG.get('channel_range', (0, num_channels))
    ch_end = min(ch_end, num_channels)
    for channel_idx in range(ch_start, ch_end):
        train_loader, val_loader = create_data_loaders_for_channel(channel_idx, train_paths, val_paths, current_layer)
        model = create_model_for_channel(input_dim_per_channel, latent_dim)
        criterion = nn.MSELoss()
        optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
        scheduler = OneCycleLR(optimizer, max_lr=CONFIG['lr'], epochs=CONFIG['epochs'], steps_per_epoch=len(train_loader))

        train_losses_channel = []
        val_losses_channel = []
        best_val_loss_channel = float("inf")

        for epoch in range(CONFIG['epochs']):
            train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, device, CONFIG['topk_k'], epoch, use_mixed_precision=use_mixed_precision, dtype=dtype, scaler=scaler)
            val_loss = evaluate(model, val_loader, criterion, device, use_mixed_precision=use_mixed_precision, dtype=dtype)
            train_losses_channel.append(train_loss)
            val_losses_channel.append(val_loss)
            if val_loss < best_val_loss_channel:
                best_val_loss_channel = val_loss
                torch.save(get_state_dict_to_save(model), os.path.join(output_dir_layer, f"model_channel_{channel_idx}_best.pth"))
            if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == CONFIG['epochs'] - 1:
                print(f"  Layer {current_layer} Ch {channel_idx} Epoch {epoch+1}/{CONFIG['epochs']} | Train: {train_loss:.6f} Val: {val_loss:.6f} Best: {best_val_loss_channel:.6f}")

            # Every 100 epochs: save reconstructed channel .npy for test proteins (for structure module / TM-score later)
            if (epoch + 1) % 100 == 0 and len(test_paths) > 0:
                npb_dir = os.path.join(output_dir_layer, f"npb_epoch_{epoch+1}")
                os.makedirs(npb_dir, exist_ok=True)
                model.eval()
                ts = CONFIG['target_size']
                with torch.no_grad():
                    for path in test_paths:
                        arr = np.load(path)
                        L = arr.shape[0]
                        ch = arr[:, :, channel_idx].astype(np.float32)
                        max_val = np.abs(ch).max()
                        if max_val > 1e-8:
                            ch = ch / max_val
                        img = Image.fromarray(ch, mode='F')
                        img = img.resize((ts, ts), Image.Resampling.LANCZOS)
                        x = np.array(img).reshape(-1)
                        x_t = torch.from_numpy(x).unsqueeze(0).to(device)
                        if use_mixed_precision:
                            with autocast('cuda', dtype=dtype):
                                rec = model(x_t)
                        else:
                            rec = model(x_t)
                        rec = rec.cpu().numpy().reshape(ts, ts).astype(np.float32)
                        img_rec = Image.fromarray(rec, mode='F')
                        img_rec = img_rec.resize((L, L), Image.Resampling.LANCZOS)
                        rec_L = np.array(img_rec)
                        name = os.path.basename(path).replace(f'_pair_block_{current_layer}.npy', '')
                        np.save(os.path.join(npb_dir, f"{name}_channel_{channel_idx}.npy"), rec_L)
                model.train()
                print(f"  💾 NPB epoch {epoch+1}: saved {len(test_paths)} test reconstructions to {npb_dir}")

        torch.save(get_state_dict_to_save(model), os.path.join(output_dir_layer, f"model_channel_{channel_idx}_final.pth"))
        all_channel_results['train_losses'].append(train_losses_channel)
        all_channel_results['val_losses'].append(val_losses_channel)
        all_channel_results['best_val_losses'].append(best_val_loss_channel)
        all_channel_results['final_train_losses'].append(train_losses_channel[-1])
        all_channel_results['final_val_losses'].append(val_losses_channel[-1])
        del model, optimizer, scheduler, train_loader, val_loader
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        channels_done += 1
        elapsed = time.time() - overall_start_time
        if channels_done > 0 and total_channels_estimate > channels_done:
            eta_sec = (elapsed / channels_done) * (total_channels_estimate - channels_done)
            eta_min, eta_hr = eta_sec / 60, eta_sec / 3600
            print(f"  ⏱️  Progress: {channels_done}/{total_channels_estimate} channel runs | Elapsed: {elapsed/60:.1f} min | ETA: {eta_min:.0f} min ({eta_hr:.1f} h) remaining")

    all_results_by_layer[current_layer] = {k: list(v) for k, v in all_channel_results.items()}
    avg_best = np.mean(all_channel_results['best_val_losses'])
    print(f"\n✅ Layer {current_layer} done — avg best val loss: {avg_best:.8f}, models in {output_dir_layer}")

total_time = time.time() - overall_start_time
# Keep last layer's results in all_channel_results for plot/save cells
if all_results_by_layer:
    last_layer = max(all_results_by_layer.keys())
    all_channel_results = {k: list(v) for k, v in all_results_by_layer[last_layer].items()}
else:
    all_channel_results = {'train_losses': [], 'val_losses': [], 'best_val_losses': [], 'final_train_losses': [], 'final_val_losses': []}

print("\n" + "="*80)
print("ALL LAYERS AND CHANNELS COMPLETE!")
print("="*80)
print(f"Layers trained: {list(all_results_by_layer.keys())}")
print(f"Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"✅ Models saved under {CONFIG['output_dir']}/layer_47/ (NPB every 100 epochs in npb_epoch_*)")
print("="*80)

In [ ]:
# Cell 12: Plot Losses (Average Across All Channels — last layer only)
# Plots use all_channel_results from the last trained layer
if not all_channel_results['train_losses']:
    print("No training results to plot (no layers trained).")
else:
    avg_train_losses = np.mean(all_channel_results['train_losses'], axis=0)
    avg_val_losses = np.mean(all_channel_results['val_losses'], axis=0)
    plt.figure(figsize=(12, 6))
    plt.plot(avg_train_losses, label="Avg Train Loss", marker="o", markersize=3, linewidth=1.5)
    plt.plot(avg_val_losses, label="Avg Val Loss", marker="s", markersize=3, linewidth=1.5)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss (MSE)", fontsize=12)
    plt.title(f"Sparse Autoencoder Loss — Last Layer, Avg Across {num_channels} Channels", fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.yscale("log")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_dir'], "losses_avg.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Average loss plot saved to {CONFIG['output_dir']}/losses_avg.png")
    plt.figure(figsize=(14, 6))
    channels_list = list(range(num_channels))
    plt.plot(channels_list, all_channel_results['best_val_losses'], marker='o', markersize=4, linewidth=1.5, label='Best Val Loss per Channel')
    plt.xlabel("Channel Index", fontsize=12)
    plt.ylabel("Best Validation Loss (MSE)", fontsize=12)
    plt.title(f"Best Validation Loss for Each Channel — Last Layer ({num_channels} channels)", fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.yscale("log")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_dir'], "losses_per_channel.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Per-channel loss plot saved to {CONFIG['output_dir']}/losses_per_channel.png")

In [ ]:
# Cell 13: Save Training Info (All Layers + Channels)
info = {
    "architecture": "Sparse Autoencoder (SAE) - Channel-by-Channel, All Layers 0..47",
    "layers": CONFIG['layers'],
    "layers_trained": list(all_results_by_layer.keys()) if all_results_by_layer else [],
    "avg_best_val_loss_per_layer": {str(l): float(np.mean(all_results_by_layer[l]['best_val_losses'])) for l in all_results_by_layer} if all_results_by_layer else {},
    "input_dim_per_channel": input_dim_per_channel,
    "latent_dim": latent_dim,
    "input_shape_per_channel": [1, CONFIG['target_size'], CONFIG['target_size']],
    "encoder_hidden": CONFIG['encoder_hidden'],
    "decoder_hidden": CONFIG['decoder_hidden'],
    "topk_k": CONFIG['topk_k'],
    "topk_fraction": CONFIG['topk_fraction'],
    "use_batch_norm": CONFIG['use_batch_norm'],
    "use_weight_tying": CONFIG['use_weight_tying'],
    "num_channels": num_channels,
    "best_val_losses_per_channel": all_channel_results['best_val_losses'],
    "final_train_losses_per_channel": all_channel_results['final_train_losses'],
    "final_val_losses_per_channel": all_channel_results['final_val_losses'],
    "avg_best_val_loss": float(np.mean(all_channel_results['best_val_losses'])) if all_channel_results['best_val_losses'] else None,
    "avg_final_train_loss": float(np.mean(all_channel_results['final_train_losses'])) if all_channel_results['final_train_losses'] else None,
    "avg_final_val_loss": float(np.mean(all_channel_results['final_val_losses'])) if all_channel_results['final_val_losses'] else None,
    "epochs": CONFIG['epochs'],
    "batch_size": CONFIG['batch_size'],
    "lr": CONFIG['lr'],
    "weight_decay": CONFIG['weight_decay'],
    "target_size": CONFIG['target_size'],
    "num_proteins": len(protein_files),
    "protein_files": [os.path.basename(pf) for pf in protein_files],
    "train_ratio": CONFIG['train_ratio'],
    "val_ratio": CONFIG['val_ratio'],
    "test_ratio": CONFIG['test_ratio'],
    "split_seed": CONFIG['split_seed'],
    "n_train": len(train_paths),
    "n_val": len(val_paths),
    "n_test": len(test_paths),
    "train_protein_ids": [os.path.basename(p).replace(f"_pair_block_{CONFIG['layer']}.npy", "") for p in train_paths],
    "test_protein_ids": [os.path.basename(p).replace(f"_pair_block_{CONFIG['layer']}.npy", "") for p in test_paths],
    "train_proteins": [os.path.basename(p) for p in train_paths],
    "val_proteins": [os.path.basename(p) for p in val_paths],
    "test_proteins": [os.path.basename(p) for p in test_paths],
    "interpolation": "Lanczos (sinc-based)",
    "mixed_precision": use_mixed_precision,
    "dtype": CONFIG['dtype'],
}

with open(os.path.join(CONFIG['output_dir'], "training_info.json"), "w") as f:
    json.dump(info, f, indent=2)

print(f"✅ Training info saved to {CONFIG['output_dir']}/training_info.json")
print("\nTraining Summary:")
print(f"  Layers trained: {info['layers_trained']}")
print(f"  Channels per layer: {num_channels}")
if all_channel_results['best_val_losses']:
    print(f"  (Last layer) Avg Best Val Loss: {info['avg_best_val_loss']:.8f}")
    print(f"  (Last layer) Best Channel: {np.argmin(all_channel_results['best_val_losses'])} (loss: {min(all_channel_results['best_val_losses']):.8f})")
    print(f"  (Last layer) Worst Channel: {np.argmax(all_channel_results['best_val_losses'])} (loss: {max(all_channel_results['best_val_losses']):.8f})")